In [1]:
# ============================================
# BLACK HOLE SIMULATION - LIVE ANIMATION
# with V_eff evaluated at every step
# ============================================

%matplotlib osx   

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle

# ============================================
# EFFECTIVE POTENTIAL  (called per step)
# ============================================

RS_NORM = 1.0
M_NORM  = 0.5

def v_eff(r, L):
    """Schwarzschild effective potential evaluated at radius r."""
    factor = max(1.0 - RS_NORM / r, 1e-9)
    return factor * (1.0 + L**2 / r**2)

def dv_eff_dr(r, L):
    """
    Analytic derivative -dV_eff/dr.
    This replaces the hardcoded r_acc expression in the original code.
    Both are mathematically identical — now explicit.
    """
    return (
        - M_NORM         / r**2
        + L**2           / r**3
        - 3 * M_NORM * L**2 / r**4
    )

# ============================================
# SIMULATION  (V_eff evaluated every step)
# ============================================

def simulate_particle(x0, y0, vx0, vy0, steps=18000, dt=1):
    r   = np.sqrt(x0**2 + y0**2)
    phi = np.arctan2(y0, x0)
    vr  = (x0 * vx0 + y0 * vy0) / r
    vphi= (x0 * vy0 - y0 * vx0) / r**2
    L   = r**2 * vphi

    # --- initial energy check against V_eff ---
    V0 = v_eff(r, L)
    factor = max(1.0 - RS_NORM / r, 0.001)
    E  = factor * (1.0 + abs(vr))
    if E**2 < V0:
        E = np.sqrt(V0) * 1.05

    path_x, path_y, path_r   = [], [], []
    path_veff, path_forbidden = [], []

    for step in range(steps):
        # ── 1. Evaluate V_eff at current position ──────────────────────
        Vc           = v_eff(r, L)
        in_forbidden = (E**2 < Vc)          # classically forbidden?

        # ── 2. Acceleration from -dV_eff/dr ────────────────────────────
        r_acc = dv_eff_dr(r, L)

        # ── 3. Integrate (unchanged from original) ──────────────────────
        phi_dot = L / r**2
        vr  += r_acc  * dt
        r   += vr     * dt
        phi += phi_dot * dt

        # ── 4. Termination checks ───────────────────────────────────────
        if r <= RS_NORM * 1.05:
            print(f"  → Absorbed at step {step}")
            break
        if r > 65 or r <= 0.01:
            print(f"  → Escaped at step {step}")
            break

        path_x.append(r * np.cos(phi))
        path_y.append(r * np.sin(phi))
        path_r.append(r)
        path_veff.append(Vc)
        path_forbidden.append(in_forbidden)

    return (np.array(path_x), np.array(path_y), np.array(path_r),
            np.array(path_veff), np.array(path_forbidden), L, E)

# ============================================
# PARTICLES  (same initial conditions)
# ============================================

bh_particles = [
    (10,   0,    -0.12345,    0.22,  'red',    'P1 - Test1'),
    (65, 0,    0,    0,     'white',  'P2 - Test2'),
    (30,   0,    0.09, -0.1,  'yellow', 'P3 - Test3'),
    (18,   0,    0.05,    -0.2,   'cyan',   'P4 - Test4'),   
    (18,   0,    -0.3, 0.3,   'hotpink',   'P5 - Test5'),
]

# ============================================
# COMPUTE PATHS
# ============================================

print("Computing particle paths with per-step V_eff...")
bh_paths = []
bh_L     = []
bh_E     = []
for p in bh_particles:
    print(f"  Simulating {p[5]}...")
    result = simulate_particle(*p[:4])
    bh_paths.append(result[:5])          # x, y, r, veff, forbidden
    bh_L.append(result[5])
    bh_E.append(result[6])
    print(f"    Path length: {len(result[0])} steps")
print("All paths computed!\n")

# ============================================
# FIGURE LAYOUT  (3 panels now)
# ============================================

fig = plt.figure(figsize=(19, 8), facecolor='black')
gs  = gridspec.GridSpec(
    2, 3,
    figure=fig,
    width_ratios=[3.2, 1.2, 1.2],
    height_ratios=[1, 1],
    hspace=0.45, wspace=0.32,
    left=0.04, right=0.97, top=0.91, bottom=0.08
)

ax      = fig.add_subplot(gs[:, 0])      # orbital view  (spans both rows)
ax_veff = fig.add_subplot(gs[0, 1])     # V_eff(r) curves
ax_grav = fig.add_subplot(gs[0, 2])     # gravity strength meter (original)
ax_r    = fig.add_subplot(gs[1, 1])     # r(t) traces
ax_forb = fig.add_subplot(gs[1, 2])     # forbidden-region flag

DARK = '#03040c'
DIM  = '#1a1f3a'

def style(a, title):
    a.set_facecolor(DARK)
    for sp in a.spines.values(): sp.set_edgecolor(DIM)
    a.tick_params(colors='#4a5070', labelsize=7)
    a.grid(True, color=DIM, linewidth=0.6)
    a.set_title(title, color='#c8cfe8', fontsize=8, pad=4)

# ── Main orbital panel ──────────────────────────────────────────────────────
ax.set_facecolor('black')
ax.set_xlim(-55, 55)
ax.set_ylim(-50, 50)
ax.set_aspect('equal')
ax.set_title('Orbital Stability and Capture Dynamics Near a Black Hole\n'
             'RED trail = particle in V_eff forbidden region (E² < V_eff)',
             color='white', fontsize=11)
ax.set_xlabel('X (rs)', color='white')
ax.set_ylabel('Y (rs)', color='white')
ax.tick_params(colors='white')
ax.grid(True, alpha=0.1, color='white')

# Event horizon
ax.add_patch(Circle((0,0), 5.0, color='red', fill=False,
                    linewidth=2, linestyle='--', label='Event horizon (5rₛ)'))
# Singularity
ax.add_patch(Circle((0,0), 1.0, color='white', fill=True))
# Glow
for rad, alp in [(8, 0.04), (6, 0.08), (5.2, 0.14)]:
    ax.add_patch(Circle((0,0), rad, color='orange', alpha=alp))

ax.plot([], [], '-', color='#FF4444', lw=1.5, label='Forbidden region (E²<Veff)')
ax.legend(facecolor='black', labelcolor='white', loc='upper right', fontsize=8)

# ── V_eff curves panel ──────────────────────────────────────────────────────
style(ax_veff, 'V_eff(r) per particle')
r_arr = np.linspace(RS_NORM * 1.12, 38, 500)
for p, L, E in zip(bh_particles, bh_L, bh_E):
    Vc = np.array([v_eff(rr, L) for rr in r_arr])
    ax_veff.plot(r_arr, Vc, color=p[4], lw=1.2, alpha=0.8, label=p[5])
    ax_veff.axhline(E**2, color=p[4], lw=0.7, linestyle='--', alpha=0.5)
ax_veff.axvline(RS_NORM, color='#FF2244', lw=0.9, linestyle=':', alpha=0.7)
ax_veff.set_ylim(0, 4.5)
ax_veff.set_xlabel('r (rₛ)', color='#4a5070', fontsize=7)
ax_veff.set_ylabel('V_eff', color='#4a5070', fontsize=7)
ax_veff.legend(facecolor=DARK, labelcolor='#c8cfe8', fontsize=5, loc='upper right')

# ── Gravity meter (original right panel, restyled) ──────────────────────────
style(ax_grav, 'Closest approach & gravity')
ax_grav.set_xlim(0, 1.05)
ax_grav.set_ylim(0, 25)
ax_grav.set_xlabel('Normalised gravity', color='#4a5070', fontsize=7)
ax_grav.set_ylabel('r (rₛ)',             color='#4a5070', fontsize=7)
gravity_dot,  = ax_grav.plot([], [], 'o', color='white', ms=10, zorder=5)
gravity_text  = ax_grav.text(0.05, 18, '', color='white', fontsize=8)

# ── r(t) panel ──────────────────────────────────────────────────────────────
style(ax_r, 'r(t) — radial distance')
ax_r.set_xlabel('Step', color='#4a5070', fontsize=7)
ax_r.set_ylabel('r (rₛ)', color='#4a5070', fontsize=7)
ax_r.axhline(RS_NORM, color='#FF2244', lw=0.8, linestyle='--', alpha=0.6)
r_lines = [ax_r.plot([], [], '-', color=p[4], lw=1.0, alpha=0.8)[0]
           for p in bh_particles]

# ── Forbidden-region panel ───────────────────────────────────────────────────
style(ax_forb, 'V_eff at current position')
ax_forb.set_xlabel('Step', color='#4a5070', fontsize=7)
ax_forb.set_ylabel('V_eff(r)',  color='#4a5070', fontsize=7)
veff_lines = [ax_forb.plot([], [], '-', color=p[4], lw=1.0, alpha=0.8)[0]
              for p in bh_particles]
# E² reference lines
for p, E in zip(bh_particles, bh_E):
    ax_forb.axhline(E**2, color=p[4], lw=0.6, linestyle=':', alpha=0.4)

fig.suptitle('BLACK HOLE SIMULATION  ·  Effective Potential V_eff Evaluated Per Step',
             color='#FF6B00', fontsize=12, fontweight='bold')

import warnings
warnings.filterwarnings('ignore', message='.*tight_layout.*')
plt.tight_layout()
plt.tight_layout()
plt.ion()
plt.show()

# ============================================
# ANIMATION LOOP  (same structure as original)
# ============================================

trail_length = 100000
total_frames = 300

# Pre-build coloured segments per particle for forbidden-region colouring
# We use two overlapping trail lines: normal colour + red override

# Two trail lines per particle: base colour + forbidden override
bh_trails_base = [ax.plot([], [], '-', color=p[4], lw=1.5, alpha=0.85)[0]
                  for p in bh_particles]
bh_trails_forb = [ax.plot([], [], '-', color='#FF3333', lw=2.0, alpha=0.9)[0]
                  for p in bh_particles]
bh_dots         = [ax.plot([], [], 'o', color=p[4], ms=8)[0]
                   for p in bh_particles]
bh_labels       = [ax.text(0, 0, p[5], color=p[4], fontsize=8)
                   for p in bh_particles]

print("Animation running...")

for frame in range(total_frames):
    closest_r = 999

    for i, (path, trail_b, trail_f, dot, label) in enumerate(
            zip(bh_paths, bh_trails_base, bh_trails_forb, bh_dots, bh_labels)):

        px, py, pr, pveff, pfbd = path
        n = len(px)
        if n < 2:
            continue

        idx   = min(int((frame / total_frames) * n), n - 1)
        start = max(0, idx - trail_length)

        # ── Base trail (full colour) ────────────────────────────────────
        trail_b.set_data(px[start:idx], py[start:idx])

        # ── Forbidden-region trail overlay (red) ────────────────────────
        fbd_seg_x, fbd_seg_y = [], []
        for j in range(start, idx):
            if pfbd[j]:
                fbd_seg_x.append(px[j])
                fbd_seg_y.append(py[j])
            else:
                fbd_seg_x.append(np.nan)
                fbd_seg_y.append(np.nan)
        trail_f.set_data(fbd_seg_x, fbd_seg_y)

        dot.set_data([px[idx]], [py[idx]])
        label.set_position((px[idx] + 0.8, py[idx] + 0.8))

        if pr[idx] < closest_r:
            closest_r = pr[idx]

        # ── r(t) panel ──────────────────────────────────────────────────
        r_lines[i].set_data(np.arange(start, idx), pr[start:idx])

        # ── V_eff(t) panel ───────────────────────────────────────────────
        veff_lines[i].set_data(np.arange(start, idx), pveff[start:idx])

    # ── Gravity meter (original logic) ──────────────────────────────────
    if closest_r < 999:
        g_strength = min(1.0 / closest_r**2, 1.0)
        g_norm     = min(g_strength / (1.0 / 0.5**2), 1.0)
        gravity_dot.set_data([g_norm], [closest_r])
        gravity_text.set_text(
            f'Closest: {closest_r:.1f} rs\n'
            f'Gravity: {g_strength:.3f}'
        )

    # Dynamic axis limits for r(t) and V_eff(t) panels
    ax_r.set_xlim(0, max(len(p[0]) for p in bh_paths))
    ax_r.set_ylim(0, 40)
    ax_forb.set_xlim(0, max(len(p[0]) for p in bh_paths))
    ax_forb.set_ylim(0, 3.5)

    # ── Moving dots on V_eff curve panel ────────────────────────────────
    for dot_on_curve in getattr(fig, '_veff_dots', []):
        try: dot_on_curve.remove()
        except: pass
    fig._veff_dots = []
    for i, (path, p, L) in enumerate(zip(bh_paths, bh_particles, bh_L)):
        px, py, pr, pveff, pfbd = path
        n = len(px)
        if n < 2: continue
        idx = min(int((frame / total_frames) * n), n - 1)
        cur_r    = pr[idx]
        cur_veff = pveff[idx]
        if RS_NORM * 1.12 <= cur_r <= 38:
            d, = ax_veff.plot(cur_r, cur_veff, 'o', color=p[4], ms=5, zorder=6)
            fig._veff_dots.append(d)

    fig.canvas.draw()
    fig.canvas.flush_events()
    plt.pause(0.05)

plt.ioff()
print("Animation complete!")

Computing particle paths with per-step V_eff...
  Simulating P1 - Test1...
    Path length: 18000 steps
  Simulating P2 - Test2...
  → Absorbed at step 821
    Path length: 821 steps
  Simulating P3 - Test3...
    Path length: 18000 steps
  Simulating P4 - Test4...
    Path length: 18000 steps
  Simulating P5 - Test5...
  → Escaped at step 206
    Path length: 206 steps
All paths computed!

Animation running...
Animation complete!
